# §10 Sensitivity Analysis — 决策稳定性

Drop-in analysis after §9. Reads the 24h / 48h prediction CSVs and tests:

1. **Prediction perturbation** — δ ∈ {±10%, ±30%} via two schemes:
   - Stochastic: multiplicative Gaussian noise, `N_TRIALS` trials per cell
   - Deterministic: uniform scaling `pred × (1 + δ)`
2. **Blend weight sweep** — `w ∈ {0, 0.25, 0.5, 0.75, 1.0}`
3. **Output** — 3-panel heatmap + county robustness table + per-cell picks table

**Inputs:** `results/direct_pred_24h.csv`, `results/direct_pred_48h.csv` (produced by §8)

**Outputs:**
- `results/sensitivity_heatmap.png`
- `results/sensitivity_county_freq.csv`
- `results/uniform_picks_detail.csv`
- `results/blend_sweep_picks.csv`


In [ ]:
# ---- Config ----
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter

NUM_GENERATORS = 5
GEN_CAPACITY   = 1000
N_TRIALS       = 100              # Monte-Carlo trials per (delta, w)
RNG            = np.random.default_rng(42)

PRED_24H_CSV = 'results/direct_pred_24h.csv'
PRED_48H_CSV = 'results/direct_pred_48h.csv'

os.makedirs('results', exist_ok=True)

## 1. Load predictions from CSV, pivot to (T, L) matrices

In [ ]:
def load_pred_csv(path, expected_T):
    df = pd.read_csv(path)
    assert set(df.columns) >= {'timestamp', 'location', 'pred'}, \
        f"{path} 列不对: {df.columns.tolist()}"
    df['timestamp'] = pd.to_datetime(df['timestamp'])
    df['location']  = df['location'].astype(str)            # 统一 FIPS 为 str
    mat_df = df.pivot(index='timestamp', columns='location', values='pred')
    mat_df = mat_df.sort_index()                            # 时间升序
    mat_df = mat_df.reindex(sorted(mat_df.columns), axis=1) # county 字典序
    assert mat_df.shape[0] == expected_T, \
        f"{path} 时间步 {mat_df.shape[0]} != 期望 {expected_T}"
    assert not mat_df.isna().any().any(), f"{path} 有 NaN, 检查 CSV 是否完整"
    return mat_df.values.astype(np.float64), list(mat_df.columns), list(mat_df.index)

pred_24h_mat, locs_24, ts_24 = load_pred_csv(PRED_24H_CSV, expected_T=24)
pred_48h_mat, locs_48, ts_48 = load_pred_csv(PRED_48H_CSV, expected_T=48)

# 两个文件必须用同一批县, 否则 blend 无意义
assert locs_24 == locs_48, "24h/48h CSV county 列表不一致, 检查上游"
locs = locs_48
L    = len(locs)
print(f"pred_24h_mat: {pred_24h_mat.shape} (peak={pred_24h_mat.max():.1f})")
print(f"pred_48h_mat: {pred_48h_mat.shape} (peak={pred_48h_mat.max():.1f})")
print(f"县数: {L}, 前 24h 时间窗: {ts_48[0]} .. {ts_48[23]}")

## 2. Greedy allocator + blend + overlap metrics

In [ ]:
def greedy_allocate(pred_mat, n_gen=NUM_GENERATORS, cap=GEN_CAPACITY):
    """§9 贪心分配器的向量化版本. 返回 list[int] (county indices)."""
    L_ = pred_mat.shape[1]
    alloc = np.zeros(L_, dtype=int)
    picks = []
    for _ in range(n_gen):
        # 一次性对所有 county 算加一台的增量缓解
        cur_caps = (alloc      * cap)[None, :]
        nxt_caps = ((alloc+1) * cap)[None, :]
        gains = (np.minimum(pred_mat, nxt_caps).sum(axis=0)
                 - np.minimum(pred_mat, cur_caps).sum(axis=0))
        best_i = int(np.argmax(gains))
        alloc[best_i] += 1
        picks.append(best_i)
    return picks

def build_blended(w, p24, p48):
    """§9 blend 规则: 前 24h 混合, 后 24h 保持 pred_48h."""
    out = p48.copy()
    out[:24] = w * p24 + (1.0 - w) * p48[:24]
    return np.clip(out, 0, None)

def multiset_overlap(a, b):
    """两个 allocation 的 multiset 交集 (0 ~ NUM_GENERATORS)."""
    ca, cb = Counter(a), Counter(b)
    return sum(min(ca[k], cb[k]) for k in ca)

## 3. Baseline decision (w = 0.5, no noise)

In [ ]:
base_blended   = build_blended(0.5, pred_24h_mat, pred_48h_mat)
baseline_picks = greedy_allocate(base_blended)
baseline_fips  = [locs[i] for i in baseline_picks]
print(f"Baseline (w=0.5, no noise): {baseline_fips}")

deltas = [-0.30, -0.10, 0.0, 0.10, 0.30]
ws     = [0.0, 0.25, 0.5, 0.75, 1.0]

## 4. Scan A — Stochastic (multiplicative Gaussian noise)

For each `(δ, w)` cell, draw `N_TRIALS` samples of noisy predictions and re-run greedy.
Track mean overlap with baseline and exact-match probability.


In [ ]:
mean_overlap = np.zeros((len(deltas), len(ws)))
exact_match  = np.zeros((len(deltas), len(ws)))
county_freq  = Counter()
total_trials = 0

for i, delta in enumerate(deltas):
    for j, w in enumerate(ws):
        blended = build_blended(w, pred_24h_mat, pred_48h_mat)
        sigma   = abs(delta) + 0.05           # 保证 δ=0 时也有基础噪声
        overlaps, exacts = [], []
        for _ in range(N_TRIALS):
            noise     = RNG.normal(1.0 + delta, sigma, size=blended.shape)
            perturbed = np.clip(blended * noise, 0, None)
            picks     = greedy_allocate(perturbed)
            ov        = multiset_overlap(picks, baseline_picks)
            overlaps.append(ov)
            exacts.append(int(ov == NUM_GENERATORS))
            for idx in picks:
                county_freq[locs[idx]] += 1
            total_trials += 1
        mean_overlap[i, j] = np.mean(overlaps)
        exact_match[i, j]  = np.mean(exacts)

print(f"完成 Scan A: {total_trials} 次随机 trials")

## 5. Scan B — Deterministic uniform scaling

Uniform scaling `pred × (1 + δ)` — no randomness, so one greedy call per cell.
The capacity cap (1,000) breaks scale-invariance, making this test informative.


In [ ]:
uniform_overlap = np.zeros((len(deltas), len(ws)), dtype=int)
uniform_picks_table = []
for i, delta in enumerate(deltas):
    for j, w in enumerate(ws):
        blended = build_blended(w, pred_24h_mat, pred_48h_mat) * (1.0 + delta)
        blended = np.clip(blended, 0, None)
        picks   = greedy_allocate(blended)
        uniform_overlap[i, j] = multiset_overlap(picks, baseline_picks)
        uniform_picks_table.append({
            'delta'  : delta,
            'w'      : w,
            'picks'  : [locs[p] for p in picks],
            'overlap': uniform_overlap[i, j],
        })

uniform_df = pd.DataFrame(uniform_picks_table)
uniform_df.to_csv('results/uniform_picks_detail.csv', index=False)

# Show only cells where the decision flips
changed = uniform_df[uniform_df['overlap'] != NUM_GENERATORS]
print("--- Cells where decision DIFFERS from baseline ---")
print(changed.to_string(index=False) if len(changed) else "  (none — decision stable across full grid)")
print(f"\nTotal cells: {len(uniform_df)}, changed: {len(changed)}")

## 6. Heatmap visualization

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

def _heatmap(ax, mat, title, vmin, vmax, cmap, fmt=".2f"):
    im = ax.imshow(mat, cmap=cmap, vmin=vmin, vmax=vmax, aspect='auto')
    ax.set_xticks(range(len(ws)));     ax.set_xticklabels([f"w={w}" for w in ws])
    ax.set_yticks(range(len(deltas))); ax.set_yticklabels([f"{int(d*100):+d}%" for d in deltas])
    ax.set_xlabel("Blend weight w")
    ax.set_ylabel("Prediction perturbation δ")
    ax.set_title(title)
    for i in range(mat.shape[0]):
        for j in range(mat.shape[1]):
            v = mat[i, j]
            color = 'white' if (v - vmin) / (vmax - vmin + 1e-9) < 0.45 else 'black'
            ax.text(j, i, format(v, fmt), ha='center', va='center', color=color, fontsize=9)
    return im

im1 = _heatmap(axes[0], mean_overlap,
               f"Mean overlap @ 5 (noisy pred, N={N_TRIALS})",
               vmin=0, vmax=5, cmap='viridis', fmt=".2f")
plt.colorbar(im1, ax=axes[0], label='Overlap / 5')

im2 = _heatmap(axes[1], exact_match,
               f"P(exact multiset match) (noisy pred)",
               vmin=0, vmax=1, cmap='RdYlGn', fmt=".2f")
plt.colorbar(im2, ax=axes[1], label='Probability')

im3 = _heatmap(axes[2], uniform_overlap.astype(float),
               "Overlap @ 5 (uniform scaling, deterministic)",
               vmin=0, vmax=5, cmap='viridis', fmt=".0f")
plt.colorbar(im3, ax=axes[2], label='Overlap / 5')

plt.tight_layout()
plt.savefig('results/sensitivity_heatmap.png', dpi=120, bbox_inches='tight')
plt.show()

## 7. County robustness table

Across all `N_TRIALS × len(deltas) × len(ws)` stochastic trials, how often does each county
appear in the top-5? Counties with `freq_% > 80` are "always picked" — the robust recommendations.
(Values can exceed 100% because repeats are allowed: 2 generators at the same county → count of 2 per trial.)


In [ ]:
freq_df = pd.DataFrame([
    {'FIPS': fips, 'count': cnt,
     'freq_%': 100.0 * cnt / total_trials,
     'in_baseline': fips in baseline_fips}
    for fips, cnt in county_freq.most_common()
])
freq_df.to_csv('results/sensitivity_county_freq.csv', index=False)

print("--- County robustness (top 15) ---")
print(freq_df.head(15).to_string(index=False))

## 8. Pure blend sweep (no noise)

Isolates the effect of `w` alone. If all five rows are identical, the blend hyperparameter
introduces no decision risk — a strong robustness claim to make in the report.


In [ ]:
blend_rows = []
for w in ws:
    blended = build_blended(w, pred_24h_mat, pred_48h_mat)
    picks   = greedy_allocate(blended)
    fips    = [locs[p] for p in picks]
    ov      = multiset_overlap(picks, baseline_picks)
    blend_rows.append({'w': w, 'picks': fips, 'overlap_vs_baseline': ov})
    print(f"  w={w:.2f}: {fips}  (overlap={ov}/5)")

pd.DataFrame(blend_rows).to_csv('results/blend_sweep_picks.csv', index=False)

## 9. Summary — one-line findings for the report

In [ ]:
print(f"{'='*60}")
print("Sensitivity Analysis Summary")
print(f"{'='*60}")
print(f"Baseline picks           : {baseline_fips}")
print(f"Mean overlap (stochastic): min={mean_overlap.min():.2f}, "
      f"max={mean_overlap.max():.2f}, overall={mean_overlap.mean():.2f} / 5")
print(f"Uniform scaling overlap  : min={uniform_overlap.min()}, "
      f"max={uniform_overlap.max()} / 5")
robust = freq_df[freq_df['freq_%'] > 80]['FIPS'].tolist()
print(f"Robust picks (>80% freq) : {robust}")
fragile = [f for f in baseline_fips
           if f in freq_df['FIPS'].values
           and freq_df.set_index('FIPS').loc[f, 'freq_%'] < 50]
print(f"Fragile baseline picks   : {fragile}")